In [1]:
import pandas as pd

df = pd.read_csv("train.csv")

print(df.columns)
print(df.head(2))

Index(['dialog', 'act', 'emotion'], dtype='object')
                                              dialog                    act  \
0  ['Say , Jim , how about going for a few beers ...  [3 4 2 2 2 3 4 1 3 4]   
1  ['Can you do push-ups ? '\n " Of course I can ...          [2 1 2 2 1 1]   

                 emotion  
0  [0 0 0 0 0 0 4 4 4 4]  
1          [0 0 6 0 0 0]  


In [2]:
df = df.reset_index()
df = df.rename(columns={"index": "turn_id"})

In [3]:
# pairs = []

# for i in range(len(df)-1):

#     q = str(df.iloc[i]["dialog"]).strip()
#     a = str(df.iloc[i+1]["dialog"]).strip()

#     if len(q) > 1 and len(a) > 1:
#         pairs.append((q, a))

# print("TOTAL PAIRS:", len(pairs))
# print(pairs[:10])



import re

pairs = []

for i in range(len(df)):

    dialog = df.iloc[i]["dialog"]

    # очистка скобок
    dialog = dialog.strip()

    # удаляем [ ]
    dialog = dialog.replace("[", "").replace("]", "")

    # разбиваем по кавычкам и точкам
    turns = re.split(r"'\s*'", dialog)

    # чистим
    turns = [t.strip().strip("'").strip() for t in turns if len(t.strip()) > 2]

    # build pairs
    for j in range(len(turns)-1):

        q = turns[j]
        a = turns[j+1]

        if len(q) > 2 and len(a) > 2:
            pairs.append((q, a))

print("TOTAL PAIRS:", len(pairs))
print(pairs[:5])

TOTAL PAIRS: 41976
[('Say , Jim , how about going for a few beers after dinner ?', 'You know that is tempting but is really not good for our fitness .'), ('You know that is tempting but is really not good for our fitness .', 'What do you mean ? It will help us to relax . \'\n " Do you really think so ? I don\'t . It will just make us fat and act silly . Remember last time ? "\n " I guess you are right.But what shall we do ? I don\'t feel like sitting at home . "\n \' I suggest a walk over to the gym where we can play singsong and meet some of our friends . \'\n " That\'s a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome with them . "\n \' Sounds great to me ! If they are willing , we could ask them to go dancing with us.That is excellent exercise and fun , too . \'\n " Good.Let \' s go now . " \' All right .'), ('Can you do push-ups ? \'\n " Of course I can . It\'s a piece of cake ! Believe it or not , I can do 30 push-ups a minute . "\n

In [4]:
import pandas as pd
import ast

print("shape:", df.shape)
print("columns:", df.columns)

print("\nRAW SAMPLE:")
print(df.iloc[0])

print("\nTYPE OF dialog:")
print(type(df.iloc[0]["dialog"]))

print("\nVALUE OF dialog:")
print(df.iloc[0]["dialog"])

shape: (11118, 4)
columns: Index(['turn_id', 'dialog', 'act', 'emotion'], dtype='object')

RAW SAMPLE:
turn_id                                                    0
dialog     ['Say , Jim , how about going for a few beers ...
act                                    [3 4 2 2 2 3 4 1 3 4]
emotion                                [0 0 0 0 0 0 4 4 4 4]
Name: 0, dtype: object

TYPE OF dialog:
<class 'str'>

VALUE OF dialog:
['Say , Jim , how about going for a few beers after dinner ? '
 ' You know that is tempting but is really not good for our fitness . '
 ' What do you mean ? It will help us to relax . '
 " Do you really think so ? I don't . It will just make us fat and act silly . Remember last time ? "
 " I guess you are right.But what shall we do ? I don't feel like sitting at home . "
 ' I suggest a walk over to the gym where we can play singsong and meet some of our friends . '
 " That's a good idea . I hear Mary and Sally often go there to play pingpong.Perhaps we can make a foursome wi

In [5]:
lm_data = []

for q, a in pairs:
    sentence = (q + " " + a).split()

    for i in range(1, len(sentence)):
        context = " ".join(sentence[:i])
        target = sentence[i]

        lm_data.append((context, target))

print(len(lm_data))

1676929


In [ ]:
# =====================
# IMPORTS
# =====================
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from collections import Counter
import random

# =====================
# 1. BUILD LM DATA
# =====================
lm_data = []

for q, a in pairs:
    sentence = (q + " " + a).lower().split()

    for i in range(1, len(sentence)):
        context = sentence[:i]
        target = sentence[i]
        lm_data.append((context, target))
        lm_data = lm_data[:50000]   # ← ВОТ ЗДЕСЬ

print("LM samples:", len(lm_data))


# =====================
# 2. VOCAB
# =====================
class Vocab:
    def __init__(self):
        self.word2idx = {"<pad>":0, "<unk>":1}

    def build(self, data, min_freq=2):
        counter = Counter()

        for ctx, tgt in data:
            counter.update(ctx)
            counter.update([tgt])

        for w,f in counter.items():
            if f >= min_freq:
                self.word2idx[w] = len(self.word2idx)

        self.idx2word = {v:k for k,v in self.word2idx.items()}

    def encode(self, words):
        return [self.word2idx.get(w,1) for w in words]

    def decode(self, idx):
        return self.idx2word.get(idx, "<unk>")


vocab = Vocab()
vocab.build(lm_data)


# =====================
# 3. SPLIT DATA
# =====================
random.shuffle(lm_data)

split = int(len(lm_data)*0.9)
train_data = lm_data[split]
val_data = lm_data[split:]


# =====================
# 4. DATASET
# =====================
class LMDataset(Dataset):
    def __init__(self, data, vocab):
        self.data = data
        self.vocab = vocab

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        ctx, tgt = self.data[idx]

        x = self.vocab.encode(ctx)[:10]
        y = self.vocab.word2idx.get(tgt,1)

        return torch.tensor(x), torch.tensor(y)


def collate_fn(batch):
    xs, ys = zip(*batch)

    max_len = max(len(x) for x in xs)

    padded = []
    for x in xs:
        pad = [0]*(max_len - len(x))
        padded.append(x.tolist() + pad)

    return torch.tensor(padded), torch.tensor(ys)


train_loader = DataLoader(LMDataset(train_data, vocab), batch_size=128, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(LMDataset(val_data, vocab), batch_size=128, collate_fn=collate_fn)


# =====================
# 5. MODEL
# =====================
class LSTMLM(nn.Module):
    def __init__(self, vocab_size, emb=64, hidden=128):
        super().__init__()

        self.emb = nn.Embedding(vocab_size, emb)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, vocab_size)

    def forward(self, x):
        x = self.emb(x)
        out, _ = self.lstm(x)
        out = self.fc(out[:, -1, :])  # last token
        return out


model = LSTMLM(len(vocab.word2idx))


# =====================
# 6. EVALUATION
# =====================
def evaluate(model, loader, loss_fn):
    model.eval()

    total_loss = 0
    total_acc = 0

    with torch.no_grad():
        for x, y in loader:

            out = model(x)
            loss = loss_fn(out, y)

            total_loss += loss.item()

            pred = out.argmax(1)
            acc = (pred == y).float().mean()

            total_acc += acc.item()

    return total_loss / len(loader), total_acc / len(loader)


# =====================
# 7. TRAIN
# =====================
def train(model, train_loader, val_loader, epochs=3):

    opt = optim.Adam(model.parameters(), lr=0.001)
    loss_fn = nn.CrossEntropyLoss()

    for epoch in range(epochs):

        model.train()
        total_loss = 0

        for x, y in train_loader:

            out = model(x)
            loss = loss_fn(out, y)

            opt.zero_grad()
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()

            total_loss += loss.item()

        val_loss, val_acc = evaluate(model, val_loader, loss_fn)

        print(f"""
Epoch {epoch}
Train Loss: {total_loss:.3f}
Val Loss: {val_loss:.3f}
Val Acc: {val_acc:.3f}
Perplexity: {torch.exp(torch.tensor(val_loss)):.2f}
""")


# =====================
# 8. PREDICT
# =====================
def predict(model, vocab, text, k=5):

    model.eval()

    words = text.lower().split()
    x = torch.tensor([vocab.encode(words)])

    with torch.no_grad():
        out = model(x)
        probs = torch.softmax(out, dim=1)

        topk = torch.topk(probs, k)

        results = []

        for i in range(k):
            idx = topk.indices[0][i].item()
            prob = topk.values[0][i].item()
            results.append((vocab.decode(idx), prob))

    return results


# =====================
# RUN
# =====================
train(model, train_loader, val_loader, epochs=2)


# =====================
# TEST
# =====================
print(predict(model, vocab, "i am"))
print(predict(model, vocab, "how are"))
print(predict(model, vocab, "what is"))